# Test (trivial hull)

We sample several codes with trivial hull, rate $1/2$ and cyclic automorphism group, for several values of the code dimension.

We do the reduction to Graph Automorphism Problem (GIP) and average the running time (measured in seconds). To check the validity of the found automorphisms, we check whether I) each automorphism is indeed an automorphism of the code, and ii) if the determined group is indeed the cyclic group we forced at the beginning. 

Then, we consider also the built-in algorithm in Sagemath.

In [2]:
load('utils.sage')
import time
import csv

Generate several codes, measure average time

In [5]:
q = 2 #finite field size
num_test = 100 #number of tests for each value of k
do_proof = False #set to True if you want to test each of the found automorphisms

test_type = "OUR" #test only our algorithm
#test_type = "BUILTIN" #test only builtin
#test_type = "BOTH" #test both algorithms

#Prepare files for output
if (test_type == "OUR")|(test_type == "BOTH"):
    file_name_OUR = "Results/q"+str(q)+"_h0_cyclic_OUR.csv"
    #Prepare csv file
    with open(file_name_OUR, 'w', newline='') as outcsv:
        writer = csv.writer(outcsv)
        writer.writerow(["k", "Our"])

if (test_type == "BUILTIN")|(test_type == "BOTH"):
    file_name_BUILTIN = "Results/q"+str(q)+"_h0_cyclic_BUILTIN.csv"
    #Prepare csv file
    with open(file_name_BUILTIN, 'w', newline='') as outcsv:
        writer = csv.writer(outcsv)
        writer.writerow(["k", "BuiltIn"])


Fq = GF(q)

#For each value of k, sample code with automorphism group of size at least k.
#Then, check if it has trivial hull and, when this happens, compare Sage-built in function
#with our reduction to the Automorphisms Graph Problem

for k in range(3,30):

    n = 2*k

    our = 0 #average time for our solver
    sage_built_in = 0 #average time for built-in function

    num_codes = 0 #number of tested codes

    data = []

    while num_codes < num_test:

        #Generate code
        cycles, P, G = gen_code(Fq, n, k)        
        h = k-rank(G*G.transpose()) #hull dimension

        errors = 0 #set to 1 if something bad happens
        
        if h == 0:
            num_codes += 1
            
            if (test_type == "OUR")|(test_type == "BOTH"):
                #Our reduction
                start = time.time()
                
                A = G.transpose()*(G*G.transpose())^-1*G #adjacency matrix
                graph_A = Graph(A.change_ring(ZZ)) #graph
                auto_group = graph_A.automorphism_group() #solver for automorphisms of graph
                end = time.time()
    
                our += end-start
    
                #Check validity
                if do_proof:
                    initial_auto_group = PermutationGroup([decrement_cycles(cycles)], domain = [i for i in range(n)])
        
                    #Check if each of the initial automorphisms is included in the found group
                    for A in initial_auto_group:
                        if A not in auto_group:
                            errors = 1
        
                    #Check if each automorphism is indeed an automorphism of the code
                    H = codes.LinearCode(G).parity_check_matrix()
                    for A in auto_group:
                        A_matrix = A.matrix()
                        s = G*P*H.transpose()
                        for i in range(k):
                            for j in range(k):
                                if s[i,j]:
                                    errors = 1
                                    
                    #Check if some of found automorphisms were not ok
                    if errors:
                        print("############## SOMETHING WENT WRONG")

            if (test_type == "BUILTIN")|(test_type == "BOTH"):
                #Now, Sage built-in function
                C = codes.LinearCode(G)
    
                start = time.time()
    #            gens = C.automorphism_group_gens()
                gens = C.permutation_automorphism_group()
                end = time.time()
                
                sage_built_in += end-start

    #Print data
    print("Avg times: k = ",k)
    if (test_type == "OUR")|(test_type == "BOTH"):
        print("- Our = ",N(our/num_test))

        data = [k, N(our/num_test)]
        with open(file_name_OUR, 'a', newline='') as outcsv:
            writer = csv.writer(outcsv)
            writer.writerow(data)
            
    if (test_type == "BUILTIN")|(test_type == "BOTH"):
        print("- Built-in = ", N(sage_built_in/num_test))
        data = [k, N(sage_built_in/num_test)]
        with open(file_name_BUILTIN, 'a', newline='') as outcsv:
            writer = csv.writer(outcsv)
            writer.writerow(data)
    print("-------------------")      

Avg times: k =  3
- Our =  0.000423338413238525
-------------------
Avg times: k =  4
- Our =  0.000398938655853271
-------------------
Avg times: k =  5
- Our =  0.000431153774261475
-------------------
Avg times: k =  6
- Our =  0.000546064376831055
-------------------
Avg times: k =  7
- Our =  0.000533330440521240
-------------------
Avg times: k =  8
- Our =  0.000573191642761230
-------------------
Avg times: k =  9
- Our =  0.000632302761077881
-------------------
Avg times: k =  10
- Our =  0.000703444480895996
-------------------
Avg times: k =  11
- Our =  0.000715146064758301
-------------------
Avg times: k =  12
- Our =  0.000963034629821777
-------------------
Avg times: k =  13
- Our =  0.000917854309082031
-------------------
Avg times: k =  14
- Our =  0.00101490020751953
-------------------
Avg times: k =  15
- Our =  0.00117378950119019
-------------------
Avg times: k =  16
- Our =  0.00123414516448975
-------------------
Avg times: k =  17
- Our =  0.00127865791320

KeyboardInterrupt: 